# 05. Publication Figures

**목적**: 논문/보고서용 고해상도 figure 생성

**입력**: `output/checkpoints/{dataset}/09_trajectory.h5ad`  
**출력**: `output/figures/{dataset}/publication/`


In [ ]:
import sys
sys.path.insert(0, "../../")
sys.path.insert(0, "../")

import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

from modules.io import load_adata
from modules.plotting import umap_panel, marker_dotplot

# 출판용 설정
matplotlib.rcParams["font.family"] = "Arial"
matplotlib.rcParams["pdf.fonttype"] = 42   # PDF에서 폰트 임베딩
sc.settings.set_figure_params(dpi=300, facecolor="white", fontsize=10)

In [ ]:
DATASET = "human_genes_only"
CHECKPOINT_DIR = f"../../output/checkpoints/{DATASET}"
FIG_OUT_DIR = Path(f"../../output/figures/{DATASET}/publication")
FIG_OUT_DIR.mkdir(parents=True, exist_ok=True)

adata = load_adata(f"{CHECKPOINT_DIR}/09_trajectory.h5ad")
print(adata)

## Figure 1. Overview UMAP

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc.pl.umap(adata, color="cell_type", ax=axes[0], show=False,
           legend_loc="right margin", title="Cell Types", frameon=False)
sc.pl.umap(adata, color="sample", ax=axes[1], show=False,
           title="Samples", frameon=False)

plt.tight_layout()
plt.savefig(FIG_OUT_DIR / "fig1_umap_overview.pdf", bbox_inches="tight")
plt.savefig(FIG_OUT_DIR / "fig1_umap_overview.png", dpi=300, bbox_inches="tight")
plt.show()

## Figure 2. Marker Dotplot

In [ ]:
import yaml
with open("../../configs/markers.yaml") as f:
    markers = yaml.safe_load(f)

# 주요 마커만 선택하여 표시
key_markers = {
    "CD4 T": ["CD3D", "CD4", "IL7R"],
    "CD8 T": ["CD8A", "GZMB"],
    "NK": ["NKG7", "GNLY"],
    "B": ["MS4A1", "CD79A"],
    "Monocyte": ["CD14", "LYZ"],
    "DC": ["FCER1A", "CST3"],
}
key_markers = {k: [g for g in v if g in adata.var_names] for k, v in key_markers.items()}

if "cell_type" in adata.obs.columns:
    sc.pl.dotplot(adata, var_names=key_markers, groupby="cell_type",
                  save=str(FIG_OUT_DIR / "fig2_marker_dotplot.pdf"))

## Figure 3. Trajectory

In [ ]:
trajectory_colors = []
if "dpt_pseudotime" in adata.obs.columns:
    trajectory_colors.append("dpt_pseudotime")
if "cytotrace_score" in adata.obs.columns:
    trajectory_colors.append("cytotrace_score")

if trajectory_colors:
    umap_panel(
        adata,
        color=["cell_type"] + trajectory_colors,
        ncols=3,
        save=str(FIG_OUT_DIR / "fig3_trajectory.png"),
    )
else:
    print("No trajectory data found. Run notebook 04 first.")

## Figure 4. Cell proportion per sample

In [ ]:
if "sample" in adata.obs.columns and "cell_type" in adata.obs.columns:
    prop = adata.obs.groupby(["sample", "cell_type"]).size().unstack(fill_value=0)
    prop = prop.div(prop.sum(axis=1), axis=0)

    fig, ax = plt.subplots(figsize=(10, 5))
    prop.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    ax.set_xlabel("Sample")
    ax.set_ylabel("Cell proportion")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(FIG_OUT_DIR / "fig4_cell_proportion.pdf", bbox_inches="tight")
    plt.savefig(FIG_OUT_DIR / "fig4_cell_proportion.png", dpi=300, bbox_inches="tight")
    plt.show()
    
    prop.to_csv(FIG_OUT_DIR / "cell_proportion_table.csv")
else:
    print("Need 'sample' and 'cell_type' in obs")